# 09. Look at a Photo and Talk — VLM (openvino-genai)

> A VLM (Vision Language Model) takes a photo and text together and answers in text. Here we use **Gemma3 4B (INT4)**.
> Tokenizing, the generation loop and image processing are done by `openvino_genai.VLMPipeline` — in this lesson you learn **how to use the library well** and
> **how generative models behave** (settings · speed · fixing the answer format).

### What you will learn
1. How a VLM is built: image encoder + connector + language model
2. Generation settings: max length · greedy vs sampling · repetition penalty
3. Streaming and speed: **time to first token (TTFT)** and **tokens per second**
4. **Fixing the answer format** (JSON) — so code can use it easily
5. INT4 quantization and memory

### What happens inside a VLM
```
photo    ─[image encoder (ViT)]─▶ hundreds of image tokens ─[connector]─┐
question ─[tokenizer]─▶ text tokens ────────────────────────────────────┴─▶ [language model] ─▶ one next token ─┐
                                                                              ▲                                │
                                                                              └──── append and run again ◀─────┘  (repeat until the end token)
```

## 1. Load the model
- **INT4**: each weight is stored in 4 bits (0.5 bytes). 4 billion × 0.5 bytes ≈ 2GB + image encoder · other parts → the folder is about 3.5GB.
  We lose a little precision but save a lot of memory and time.
- If you give `CACHE_DIR`, loading is much faster from the second time on.
- If edge-lab (`run.bat`) is running, close it first (otherwise the model sits in memory twice).

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us); English notebooks live in notebooks/en/
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

import openvino_genai as og

VLM_DIR = MODELS / "vlm" / "gemma3-4b-int4"
size_gb = sum(f.stat().st_size for f in VLM_DIR.rglob("*") if f.is_file()) / 1e9
print(f"Model folder {size_gb:.1f} GB")
DEVICE = "GPU" if any(d.startswith("GPU") for d in core.available_devices) else "CPU"
CACHE = Path("ov-cache"); CACHE.mkdir(exist_ok=True)

t0 = time.perf_counter()
pipe = og.VLMPipeline(str(VLM_DIR), DEVICE, CACHE_DIR=str(CACHE))
print(f"Loaded: {time.perf_counter() - t0:.0f} s ({DEVICE})")

## 2. The photo

In [ ]:
PHOTO = "sample.jpg"          # photo to use when there is no webcam (same folder as this notebook)

def show(bgr, width=720):
    """Show an OpenCV image (BGR) inside the notebook."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """Take one frame from the webcam. If there is no webcam, use the PHOTO file."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # some webcams give dark frames at first
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"No webcam and no {PHOTO} — put a photo file name in PHOTO"
        print("No webcam, using the file:", PHOTO)
    return frame

frame = grab()
print("Image size (height, width, channels):", frame.shape)
show(frame)

## 3. Ask and get an answer — streaming
- The photo goes in as an `ov.Tensor`: `[1, height, width, 3]` RGB uint8. A very large photo only makes more image tokens and runs slower — a long side of 640 is about right.
- If you give a `streamer` function, it is called each time a token is made — the answer flows out bit by bit. Return `RUNNING` to keep going, `STOP` to stop.

**Generation settings**
| Setting | Meaning |
|---|---|
| `max_new_tokens` | The most tokens to make (one English word ≈ 1–2 tokens; one Korean character ≈ 1–2 tokens) |
| `do_sample=False` | Greedy: always the most likely token — the same question gives the same answer |
| `do_sample=True` + `temperature` | Sampling: pick by probability — more varied but less steady. Higher temperature = more so |
| `repetition_penalty` | Lowers the probability of tokens already used, to stop the model repeating itself |

In [ ]:
def to_tensor(bgr, max_side=640):
    h, w = bgr.shape[:2]
    s = min(1.0, max_side / max(h, w))
    if s < 1.0:
        bgr = cv2.resize(bgr, (int(w * s), int(h * s)))
    return ov.Tensor(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)[None].astype(np.uint8))

cfg = og.GenerationConfig()
cfg.max_new_tokens = 160
cfg.do_sample = False
cfg.repetition_penalty = 1.15

RUNNING = og.StreamingStatus.RUNNING if hasattr(og, "StreamingStatus") else False
first = {}
def streamer(piece):
    first.setdefault("t", time.perf_counter())       # when the first token came out
    print(piece, end="", flush=True)
    return RUNNING                                    # keep generating

def ask(bgr, question, quiet=False):
    first.clear()
    t0 = time.perf_counter()
    kw = {} if quiet else {"streamer": streamer}
    out = pipe.generate(question, image=to_tensor(bgr), generation_config=cfg, **kw) if bgr is not None \
        else pipe.generate(question, generation_config=cfg, **kw)
    t1 = time.perf_counter()
    n = len(pipe.get_tokenizer().encode(str(out)).input_ids.data[0])
    if not quiet:
        ttft = first.get("t", t1) - t0
        print(f"\n\nFirst token after {ttft:.1f} s · total {t1 - t0:.1f} s · {n} tokens · about {n / max(t1 - first.get('t', t0), 1e-6):.1f} tokens/s")
    return str(out)

answer = ask(frame, "Describe this photo in English in two sentences.")

**How to read it**
- **Time to first token (TTFT)**: the time to encode the photo and read the whole question at once. Bigger photos take longer.
- **Tokens per second**: the speed of making one token at a time after that. It depends a lot on memory bandwidth.

In [ ]:
answer = ask(frame, "List the objects in the photo, separated by commas. Do not write any explanation.")

## 4. Fixing the answer format — JSON
If code will use the answer, a **fixed format** is better than free sentences. Tell the model "JSON only", then take just the `{ … }` part from the text you get back.
Small models sometimes break the format — so also decide in code **what to do when it fails**.

In [ ]:
import json, re

def ask_json(bgr, question):
    text = ask(bgr, question + '\nAnswer with a single JSON object only. Do not write anything else.', quiet=True)
    m = re.search(r"\{.*\}", text, re.S)                # drop extra words before/after, keep only { … }
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

r = ask_json(frame, 'Answer about the photo in the format {"people": number, "objects": [names], "mood": "one word"}.')
print("Got:", r)
if r:
    print("people + 1 =", int(r.get("people", 0)) + 1, "← you can use it as a number right away")

## 5. Text only, no photo
If you give the same model no photo, you can use it like a text LLM (lesson 08, RAG, uses it this way).

In [ ]:
_ = ask(None, "Explain in one sentence what the NPU in OpenVINO is.")

## Common errors
| Symptom | Cause · fix |
|---|---|
| Stops while loading · out of memory | edge-lab is running / less than 16GB of memory → close it and try again, close other programs |
| Repeats the same words | Set `repetition_penalty` to 1.1–1.3 |
| Answer is cut off in the middle | `max_new_tokens` is too small |
| JSON is broken | A limit of small models → put an example in the prompt, and always handle failure |

## Summary
- VLM = image encoder + language model. The answer is built by adding tokens one at a time.
- Look at speed in two parts: TTFT (reading the photo and question) and tokens per second (generating).
- For answers your code will use, ask for a fixed format (JSON), and also handle the case when the model breaks it.

---
# 🛠 Mini project — Photo diary maker

**What to build**: Take a few photos with the webcam (or use the photos in the `photos/` folder). For each photo, the VLM makes a **title · description · tags** as JSON,
and you save them with the photos as a **Markdown diary file** (`diary.md`).

**What you use**: `to_tensor` · `ask_json` (from this notebook), saving images (`cv2.imwrite`)

**Steps**
1. Collect photos — use `photos/*.jpg` if there are any, otherwise take 3 with the webcam (5 seconds apart)
2. For each photo, `ask_json` → `{"title", "description", "tags"}`
3. Write the date · photo · content to `diary.md`

In [ ]:
from datetime import datetime

shots = [cv2.imread(str(p)) for p in sorted(Path("photos").glob("*.jpg"))] if Path("photos").is_dir() else []
if not shots:
    for i in range(3):
        print(f"{i + 1}/3 taking a photo ..."); shots.append(grab()); time.sleep(5)

Path("diary").mkdir(exist_ok=True)
lines = [f"# Photo diary — {datetime.now():%Y-%m-%d}", ""]
for i, img in enumerate(shots):
    t0 = time.perf_counter()
    r = ask_json(img, 'I want to write a diary entry from this photo. Answer in the format {"title": "short", "description": "two sentences", "tags": ["word", "word"]}.') \
        or {"title": "(could not make one)", "description": "", "tags": []}
    name = f"diary/{datetime.now():%H%M%S}_{i}.jpg"
    cv2.imwrite(name, img)
    lines += [f"## {r.get('title', '')}", f"![]({name})", "", str(r.get("description", "")), "",
              " ".join(f"#{t}" for t in r.get("tags", [])), ""]
    print(f"{i + 1}. {r.get('title')}  ({time.perf_counter() - t0:.1f} s)")

Path("diary.md").write_text("\n".join(lines), encoding="utf-8")
print("Saved: diary.md — open it from the file list on the left to see it with the photos")

**Try more**
- Add a "tag list" at the end of the diary that groups the photos by tag
- Look at several photos together and write one paragraph "summary of today" (collect the description of each photo into text and call `ask(None, ...)`)